# OpenCV bounding-box tuner

Interactive sliders for tuning the OpenCV pipeline that produces answer/line
bounding boxes.

In [1]:
import glob
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display

DATA_DIR = Path("data")
image_paths = sorted(glob.glob(str(DATA_DIR / "*.png"))) + sorted(glob.glob(str(DATA_DIR / "*.jpg")))
assert image_paths, f"No images found in {DATA_DIR.resolve()}"
print("Found images:", image_paths)

Found images: ['data/page_001.png']


## Pipeline

In [2]:
def preprocess(gray: np.ndarray, p: dict) -> np.ndarray:
    """grayscale -> denoise -> CLAHE -> deskew -> adaptive threshold -> morph cleanup -> ink mask."""
    denoised = cv2.fastNlMeansDenoising(gray, None, h=p["denoise_h"], templateWindowSize=7, searchWindowSize=21)

    clahe = cv2.createCLAHE(clipLimit=p["clahe_clip_limit"], tileGridSize=(p["clahe_grid"], p["clahe_grid"]))
    normalized = clahe.apply(denoised)

    _, otsu_inverse = cv2.threshold(normalized, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    ys, xs = np.nonzero(otsu_inverse)
    angle = 0.0
    if len(xs) >= 200:
        points = np.column_stack((xs, ys)).astype(np.float32)
        angle = cv2.minAreaRect(points)[-1]
        if angle < -45:
            angle = 90 + angle
        angle = -angle
        limit = p["max_deskew_deg"]
        angle = float(max(-limit, min(limit, angle)))

    if abs(angle) >= 0.05:
        h, w = normalized.shape[:2]
        m = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        deskewed = cv2.warpAffine(normalized, m, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    else:
        deskewed = normalized

    block_size = p["adaptive_block_size"]
    if block_size % 2 == 0:
        block_size += 1
    binary = cv2.adaptiveThreshold(
        deskewed, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
        blockSize=block_size, C=p["adaptive_c"],
    )
    ink = cv2.bitwise_not(binary)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (p["morph_close_size"], p["morph_close_size"]))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (p["morph_open_size"], p["morph_open_size"]))
    ink = cv2.morphologyEx(ink, cv2.MORPH_CLOSE, kernel_close)
    ink = cv2.morphologyEx(ink, cv2.MORPH_OPEN, kernel_open)
    return ink


def remove_ruled_lines(ink: np.ndarray, p: dict) -> np.ndarray:
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (p["rule_kernel_width"], 1))
    rules = cv2.morphologyEx(ink, cv2.MORPH_OPEN, kernel)
    rules = cv2.dilate(rules, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 3)))
    return cv2.subtract(ink, rules)


def remove_margin_lines(ink: np.ndarray, p: dict) -> np.ndarray:
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, p["margin_kernel_height"]))
    margins = cv2.morphologyEx(ink, cv2.MORPH_OPEN, kernel)
    margins = cv2.dilate(margins, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 1)))
    return cv2.subtract(ink, margins)


def clean_mask(ink: np.ndarray, p: dict) -> np.ndarray:
    cleaned = remove_margin_lines(remove_ruled_lines(ink, p), p)
    height, width = cleaned.shape[:2]
    left = min(p["page_crop_left"], width)
    right = min(p["page_crop_right"], width)
    top = min(p["page_crop_top"], height)
    bottom = min(p["page_crop_bottom"], height)
    if left or right or top or bottom:
        cleaned = cleaned.copy()
        if left:
            cleaned[:, :left] = 0
        if right:
            cleaned[:, width - right:] = 0
        if top:
            cleaned[:top, :] = 0
        if bottom:
            cleaned[height - bottom:, :] = 0
    return cleaned


def _line_from_rows(ink: np.ndarray, y0: int, y1: int) -> dict:
    band = ink[y0:y1 + 1, :]
    cols = np.nonzero(band.any(axis=0))[0]
    x0, x1 = int(cols.min()), int(cols.max())
    ink_area = int((band > 0).sum())
    return {"x": x0, "y": y0, "width": x1 - x0 + 1, "height": y1 - y0 + 1, "ink_area": ink_area}


def segment_lines(ink: np.ndarray, p: dict) -> list[dict]:
    """Horizontal projection-profile segmentation into physical lines (top-to-bottom)."""
    cleaned = clean_mask(ink, p)
    height, width = cleaned.shape[:2]
    row_ink = (cleaned > 0).sum(axis=1)
    min_row_ink = max(p["min_row_ink"], 1)

    lines: list[dict] = []
    run_start = None
    gap = 0
    for y in range(height + 1):
        active = y < height and row_ink[y] >= min_row_ink
        if active:
            if run_start is None:
                run_start = y
            gap = 0
        else:
            if run_start is not None:
                gap += 1
                if gap > p["line_gap_tolerance"]:
                    lines.append(_line_from_rows(cleaned, run_start, y - gap))
                    run_start = None
                    gap = 0
    if run_start is not None:
        lines.append(_line_from_rows(cleaned, run_start, height - 1))

    lines = [ln for ln in lines if ln["ink_area"] >= p["min_line_ink_area"]]
    lines.sort(key=lambda ln: ln["y"])
    return lines


def median_line_height(lines: list[dict]) -> float:
    if not lines:
        return 0.0
    heights = sorted(ln["height"] for ln in lines)
    return float(heights[len(heights) // 2])


def _is_header_line(line: dict, median_h: float, p: dict) -> bool:
    return line["height"] > p["header_height_factor"] * median_h


def _full_width(lines: list[dict]) -> float:
    widths = sorted(ln["width"] for ln in lines)
    if not widths:
        return 0.0
    return float(widths[len(widths) // 2])


def _union(bboxes: list[dict]) -> dict:
    x0 = min(b["x"] for b in bboxes)
    y0 = min(b["y"] for b in bboxes)
    x1 = max(b["x"] + b["width"] for b in bboxes)
    y1 = max(b["y"] + b["height"] for b in bboxes)
    return {"x": x0, "y": y0, "width": x1 - x0, "height": y1 - y0}


def detect_questions(lines: list[dict], p: dict) -> list[dict]:
    """Spatial-only re-implementation of detect_questions (no OCR text signal available here):
    first content line of the page always starts a question, matching the `index == 0`
    branch in questions.py — later lines only start a new question with OCR text support,
    which this offline notebook does not have."""
    median_h = median_line_height(lines)
    if median_h <= 0:
        return []
    content = [ln for ln in lines if not _is_header_line(ln, median_h, p)]
    if not content:
        return []

    full_w = _full_width(content)
    questions: list[dict] = []
    index = 0
    q_counter = 0
    while index < len(content):
        is_question = index == 0
        if not is_question:
            index += 1
            continue

        group = [content[index]]
        cursor = index + 1
        while cursor < len(content):
            nxt = content[cursor]
            gap = nxt["y"] - (group[-1]["y"] + group[-1]["height"])
            is_short = nxt["width"] < p["short_line_ratio"] * full_w
            if gap <= p["group_max_gap"] and is_short:
                group.append(nxt)
                cursor += 1
            else:
                break

        q_counter += 1
        questions.append({"id": f"q{q_counter:03d}", "bbox": _union(group), "lines": len(group)})
        index = cursor
    return questions


def handwriting_density(ink: np.ndarray, bbox: dict) -> float:
    """Ink pixels / bbox area, clamped to [0, 1]."""
    h, w = ink.shape[:2]
    x0 = max(0, bbox["x"])
    y0 = max(0, bbox["y"])
    x1 = min(w, bbox["x"] + bbox["width"])
    y1 = min(h, bbox["y"] + bbox["height"])
    area = max(1, (x1 - x0) * (y1 - y0))
    return float((ink[y0:y1, x0:x1] > 0).sum() / area)


def detect_answer(lines: list[dict], questions: list[dict], ink: np.ndarray, p: dict) -> dict | None:
    """Answer region: lines after the first question's bottom (and before the next
    question's top), trimmed of low-density specks and left-margin scribbles."""
    if not lines or not questions:
        return None

    first_q = min(questions, key=lambda q: q["bbox"]["y"])
    q_bottom = first_q["bbox"]["y"] + first_q["bbox"]["height"]

    later = [q for q in questions if q["bbox"]["y"] > q_bottom]
    cap = min((q["bbox"]["y"] for q in later), default=None)

    candidates = [ln for ln in lines if ln["y"] > q_bottom and (cap is None or ln["y"] < cap)]
    if not candidates:
        return None

    densities = [handwriting_density(ink, ln) for ln in candidates]
    med_density = sorted(densities)[len(densities) // 2] if densities else 0.0
    if med_density > 0:
        candidates = [ln for ln, d in zip(candidates, densities) if d >= p["trim_density_ratio"] * med_density]

    if candidates:
        lefts = sorted(float(ln["x"]) for ln in candidates)
        left_med = lefts[len(lefts) // 2]
        candidates = [
            ln for ln in candidates
            if not (ln["x"] < left_med - p["margin_scribble_px"] and ln["ink_area"] < p["min_ink_area"])
        ]

    if not candidates:
        return None

    return _union(candidates)


def answer_line_regions(lines: list[dict], answer: dict) -> list[dict]:
    """The physical lines inside the answer bbox, top to bottom."""
    return [
        ln for ln in sorted(lines, key=lambda l: l["y"])
        if answer["y"] <= ln["y"] <= answer["y"] + answer["height"]
    ]


def _normalized_weights(vg: float, ind: float, den: float) -> tuple[float, float, float]:
    total = vg + ind + den
    if total <= 0:
        return (0.55, 0.25, 0.20)
    return (vg / total, ind / total, den / total)


def detect_paragraphs(lines: list[dict], ink: np.ndarray, p: dict) -> list[dict]:
    """Split answer-region lines into paragraphs via a weighted boundary score
    over vertical gap (primary), indentation and density (secondary)."""
    ordered = sorted(lines, key=lambda ln: ln["y"])
    if not ordered:
        return []

    median_h = median_line_height(ordered) or 1.0
    left_x = min(ln["x"] for ln in ordered)
    full_w = float(max(ln["width"] for ln in ordered)) or 1.0
    w_gap, w_indent, w_density = _normalized_weights(
        p["weight_vertical_gap"], p["weight_indentation"], p["weight_density"]
    )

    boundaries: list[int] = []
    for i in range(1, len(ordered)):
        prev, cur = ordered[i - 1], ordered[i]
        gap = float(cur["y"] - (prev["y"] + prev["height"]))

        gap_score = min(1.0, max(0.0, gap / (p["gap_multiple"] * median_h)))
        indent = cur["x"] - left_x
        indent_score = min(1.0, max(0.0, indent / (0.25 * full_w)))
        prev_density = handwriting_density(ink, prev)
        cur_density = handwriting_density(ink, cur)
        density_score = min(1.0, abs(prev_density - cur_density) * 4.0)

        score = w_gap * gap_score + w_indent * indent_score + w_density * density_score
        if score >= p["paragraph_threshold"]:
            boundaries.append(i)

    starts = [0] + boundaries
    paragraphs: list[dict] = []
    for order, start in enumerate(starts, 1):
        end = starts[order] if order < len(starts) else len(ordered)
        group = ordered[start:end]
        paragraphs.append({"id": f"p{order:03d}", "bbox": _union(group)})
    return paragraphs

## Interactive tuner

In [3]:
image_dropdown = W.Dropdown(options=image_paths, description="image", layout=W.Layout(width="420px"))

def slider(desc, value, minv, maxv, step, kind=W.FloatSlider):
    return kind(value=value, min=minv, max=maxv, step=step, description=desc,
                continuous_update=False, style={"description_width": "160px"},
                layout=W.Layout(width="480px"))

# --- preprocess ---
denoise_h = slider("denoise_h", 10, 0, 30, 1, W.IntSlider)
clahe_clip_limit = slider("clahe_clip_limit", 2.0, 0.5, 6.0, 0.1)
clahe_grid = slider("clahe_grid", 8, 2, 16, 1, W.IntSlider)
max_deskew_deg = slider("max_deskew_deg", 15.0, 0, 30, 0.5)
adaptive_block_size = slider("adaptive_block_size", 31, 3, 81, 2, W.IntSlider)
adaptive_c = slider("adaptive_c", 15, -20, 30, 1, W.IntSlider)
morph_close_size = slider("morph_close_size", 3, 1, 9, 1, W.IntSlider)
morph_open_size = slider("morph_open_size", 2, 1, 9, 1, W.IntSlider)

# --- segment ---
rule_kernel_width = slider("rule_kernel_width", 60, 10, 200, 1, W.IntSlider)
margin_kernel_height = slider("margin_kernel_height", 60, 10, 200, 1, W.IntSlider)
line_gap_tolerance = slider("line_gap_tolerance", 4, 0, 30, 1, W.IntSlider)
min_row_ink = slider("min_row_ink", 12, 1, 60, 1, W.IntSlider)
min_line_ink_area = slider("min_line_ink_area", 150, 0, 1000, 10, W.IntSlider)
page_crop_left = slider("page_crop_left", 10, 0, 100, 1, W.IntSlider)
page_crop_right = slider("page_crop_right", 15, 0, 100, 1, W.IntSlider)
page_crop_top = slider("page_crop_top", 0, 0, 100, 1, W.IntSlider)
page_crop_bottom = slider("page_crop_bottom", 0, 0, 100, 1, W.IntSlider)

# --- question (anchor only — see markdown above) ---
header_height_factor = slider("header_height_factor", 4.0, 1.0, 10.0, 0.1)
group_max_gap = slider("group_max_gap", 45, 0, 200, 1, W.IntSlider)
short_line_ratio = slider("short_line_ratio", 0.7, 0.1, 1.5, 0.05)

# --- answer ---
trim_density_ratio = slider("trim_density_ratio", 0.25, 0.0, 1.0, 0.05)
margin_scribble_px = slider("margin_scribble_px", 60, 0, 200, 1, W.IntSlider)
min_ink_area = slider("min_ink_area", 800, 0, 5000, 50, W.IntSlider)

# --- paragraph ---
weight_vertical_gap = slider("weight_vertical_gap", 0.55, 0.0, 1.0, 0.05)
weight_indentation = slider("weight_indentation", 0.25, 0.0, 1.0, 0.05)
weight_density = slider("weight_density", 0.20, 0.0, 1.0, 0.05)
gap_multiple = slider("gap_multiple", 1.6, 0.5, 4.0, 0.1)
paragraph_threshold = slider("paragraph_threshold", 0.5, 0.0, 1.0, 0.05)

tunable_sliders = [
    denoise_h, clahe_clip_limit, clahe_grid, max_deskew_deg, adaptive_block_size,
    adaptive_c, morph_close_size, morph_open_size,
    rule_kernel_width, margin_kernel_height, line_gap_tolerance, min_row_ink,
    min_line_ink_area, page_crop_left, page_crop_right, page_crop_top, page_crop_bottom,
    header_height_factor, group_max_gap, short_line_ratio,
    trim_density_ratio, margin_scribble_px, min_ink_area,
    weight_vertical_gap, weight_indentation, weight_density, gap_multiple, paragraph_threshold,
]
default_values = {id(w): w.value for w in tunable_sliders}

reset_button = W.Button(description="Reset sliders", icon="undo", button_style="warning")

def reset_sliders(_):
    for w in tunable_sliders:
        w.unobserve(render, names="value")
    for w in tunable_sliders:
        w.value = default_values[id(w)]
    for w in tunable_sliders:
        w.observe(render, names="value")
    render()

reset_button.on_click(reset_sliders)

show_lines = W.Checkbox(value=True, description="show line boxes (blue)")
show_questions = W.Checkbox(value=True, description="show question boxes (purple, context)")
show_answer = W.Checkbox(value=True, description="show answer box (red)")
show_paragraphs = W.Checkbox(value=True, description="show paragraph boxes (green)")
show_ink = W.Checkbox(value=False, description="show cleaned ink mask instead of image")

out = W.Output()

def current_params():
    return dict(
        denoise_h=denoise_h.value, clahe_clip_limit=clahe_clip_limit.value, clahe_grid=clahe_grid.value,
        max_deskew_deg=max_deskew_deg.value, adaptive_block_size=adaptive_block_size.value,
        adaptive_c=adaptive_c.value, morph_close_size=morph_close_size.value, morph_open_size=morph_open_size.value,
        rule_kernel_width=rule_kernel_width.value, margin_kernel_height=margin_kernel_height.value,
        line_gap_tolerance=line_gap_tolerance.value, min_row_ink=min_row_ink.value,
        min_line_ink_area=min_line_ink_area.value, page_crop_left=page_crop_left.value,
        page_crop_right=page_crop_right.value, page_crop_top=page_crop_top.value,
        page_crop_bottom=page_crop_bottom.value, header_height_factor=header_height_factor.value,
        group_max_gap=group_max_gap.value, short_line_ratio=short_line_ratio.value,
        trim_density_ratio=trim_density_ratio.value, margin_scribble_px=margin_scribble_px.value,
        min_ink_area=min_ink_area.value,
        weight_vertical_gap=weight_vertical_gap.value, weight_indentation=weight_indentation.value,
        weight_density=weight_density.value, gap_multiple=gap_multiple.value,
        paragraph_threshold=paragraph_threshold.value,
    )

_cache = {}

def get_ink(path, p):
    key = (path, p["denoise_h"], p["clahe_clip_limit"], p["clahe_grid"], p["max_deskew_deg"],
           p["adaptive_block_size"], p["adaptive_c"], p["morph_close_size"], p["morph_open_size"])
    if key in _cache:
        return _cache[key]
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    ink = preprocess(gray, p)
    _cache.clear()  # keep memory bounded; only the latest preprocess result is needed
    _cache[key] = ink
    return ink

def render(*_):
    p = current_params()
    path = image_dropdown.value
    color = cv2.cvtColor(cv2.imread(path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    ink = get_ink(path, p)
    lines = segment_lines(ink, p)
    questions = detect_questions(lines, p)
    cleaned = clean_mask(ink, p)
    answer = detect_answer(lines, questions, cleaned, p)
    paragraphs = detect_paragraphs(answer_line_regions(lines, answer), cleaned, p) if answer else []

    with out:
        out.clear_output(wait=True)
        plt.close("all")  # self-heal any figures leaked by earlier renders in this kernel
        fig, ax = plt.subplots(figsize=(11, 11 * color.shape[0] / color.shape[1]))
        if show_ink.value:
            ax.imshow(cleaned, cmap="gray")
        else:
            ax.imshow(color)

        if show_lines.value:
            for ln in lines:
                ax.add_patch(plt.Rectangle((ln["x"], ln["y"]), ln["width"], ln["height"],
                                            fill=False, edgecolor="dodgerblue", linewidth=1))
        if show_questions.value:
            for q in questions:
                b = q["bbox"]
                ax.add_patch(plt.Rectangle((b["x"], b["y"]), b["width"], b["height"],
                                            fill=False, edgecolor="purple", linewidth=1, linestyle="--"))
                ax.text(b["x"], max(0, b["y"] - 4), q["id"], color="purple", fontsize=9, weight="bold")
        if show_answer.value and answer is not None:
            ax.add_patch(plt.Rectangle((answer["x"], answer["y"]), answer["width"], answer["height"],
                                        fill=False, edgecolor="red", linewidth=2))
            ax.text(answer["x"], max(0, answer["y"] - 4), "answer", color="red", fontsize=9, weight="bold")
        if show_paragraphs.value:
            for para in paragraphs:
                b = para["bbox"]
                ax.add_patch(plt.Rectangle((b["x"], b["y"]), b["width"], b["height"],
                                            fill=False, edgecolor="limegreen", linewidth=1.5))
                ax.text(b["x"] + b["width"] + 4, b["y"] + b["height"] / 2, para["id"],
                        color="limegreen", fontsize=8, weight="bold", va="center")

        ax.set_title(f"{Path(path).name} — {len(lines)} lines, "
                     f"{'answer box found' if answer else 'no answer box'}, "
                     f"{len(paragraphs)} paragraph(s)")
        ax.axis("off")
        plt.show()
        plt.close(fig)

controls = [
    W.HTML("<b>Preprocess</b>"), denoise_h, clahe_clip_limit, clahe_grid, max_deskew_deg,
    adaptive_block_size, adaptive_c, morph_close_size, morph_open_size,
    W.HTML("<b>Segment (line detection)</b>"), rule_kernel_width, margin_kernel_height,
    line_gap_tolerance, min_row_ink, min_line_ink_area,
    page_crop_left, page_crop_right, page_crop_top, page_crop_bottom,
    W.HTML("<b>Question grouping (anchor for answer region)</b>"),
    header_height_factor, group_max_gap, short_line_ratio,
    W.HTML("<b>Answer region</b>"), trim_density_ratio, margin_scribble_px, min_ink_area,
    W.HTML("<b>Paragraph splitting (within answer region)</b>"),
    weight_vertical_gap, weight_indentation, weight_density, gap_multiple, paragraph_threshold,
    W.HTML("<b>Display</b>"), show_lines, show_questions, show_answer, show_paragraphs, show_ink,
    reset_button,
]
for w in controls:
    if "value" in w.traits() and not isinstance(w, (W.Button, W.HTML)):
        w.observe(render, names="value")
image_dropdown.observe(render, names="value")

display(image_dropdown, W.VBox(controls), out)
render()

Dropdown(description='image', layout=Layout(width='420px'), options=('data/page_001.png',), value='data/page_0…

Output()